# Autonomous Vehicle Testing Data Analysis

This notebook analyzes the test data from various test folders (test_0 to test_4) and generates visualizations to understand:
1. Trajectories and path planning
2. Velocity profiles 
3. Cost functions
4. Behavioral planning states

The data comes from the test files used for unit testing different components of the autonomous vehicle planning system.

In [ ]:
# Import required libraries
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd

# Set some plotting parameters
plt.style.use('ggplot')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Get the path to the test folders
PATH = os.path.dirname(os.path.abspath(__file__))

In [ ]:
# Define functions for parsing the test data
def parse_trajectory_data(path):
    """Parse trajectory data from a generate_trajectory.json file"""
    with open(path) as file:
        data = json.load(file)
    
    # Extract spirals (path planning data)
    spirals = []
    for s in data.get("spirals", []):
        spiral = []
        for point in s:
            spiral.append(point)
        spirals.append(spiral)
    
    # Extract trajectory data
    trajectories = []
    for t in data.get("result", []):
        trajectory = []
        for point in t:
            trajectory.append(point)
        trajectories.append(trajectory)
    
    # Extract other data
    desired_speed = data.get("desired_speed")
    ego_state = data.get("ego_state")
    behaviour = data.get("behaviour")
    
    return spirals, desired_speed, ego_state, behaviour, trajectories

def parse_offset_goals_data(path):
    """Parse goal offset data from a generate_offset_goals.json file"""
    with open(path) as file:
        data = json.load(file)
    
    goal_input = data.get("goal_input")
    result = data.get("result", [])
    
    return goal_input, result

def parse_cost_function_data(path):
    """Parse cost function data from a cost log file"""
    with open(path) as file:
        data = json.load(file)
    
    spiral = data.get("spiral", [])
    obstacles = data.get("obstacles", [])
    goal = data.get("goal")
    collision_cost = data.get("collision_circles_cost_spiral")
    distance_cost = data.get("close_to_main_goal_cost_spiral")
    
    return spiral, obstacles, goal, collision_cost, distance_cost

def parse_state_transition_data(path):
    """Parse state transition data from a state_transition.json file"""
    with open(path) as file:
        data = json.load(file)
    
    ego_state = data.get("ego_state")
    goal_input = data.get("goal_input")
    is_junction = data.get("is_junction")
    tl_state = data.get("tl_state")
    sim_time = data.get("sim_time")
    result = data.get("result")
    
    return ego_state, goal_input, is_junction, tl_state, sim_time, result

In [ ]:
# Function to load all test data
def load_all_test_data():
    test_data = {}
    
    for test_folder in sorted(glob.glob(f"{PATH}/test_[0-9]")):
        test_name = os.path.basename(test_folder)
        test_data[test_name] = {
            "trajectory": None,
            "offset_goals": None,
            "cost_functions": [],
            "state_transition": None
        }
        
        # Load trajectory data
        trajectory_path = os.path.join(test_folder, "generate_trajectory.json")
        if os.path.exists(trajectory_path):
            test_data[test_name]["trajectory"] = parse_trajectory_data(trajectory_path)
        
        # Load offset goals data
        offset_goals_path = os.path.join(test_folder, "generate_offset_goals.json")
        if os.path.exists(offset_goals_path):
            test_data[test_name]["offset_goals"] = parse_offset_goals_data(offset_goals_path)
        
        # Load cost function data
        cost_log_folder = os.path.join(test_folder, "cost_log")
        if os.path.exists(cost_log_folder):
            for cost_file in os.listdir(cost_log_folder):
                cost_path = os.path.join(cost_log_folder, cost_file)
                test_data[test_name]["cost_functions"].append({
                    "file": cost_file,
                    "data": parse_cost_function_data(cost_path)
                })
        
        # Load state transition data
        state_transition_path = os.path.join(test_folder, "state_transition.json")
        if os.path.exists(state_transition_path):
            test_data[test_name]["state_transition"] = parse_state_transition_data(state_transition_path)
    
    return test_data

# Load all the test data
test_data = load_all_test_data()

# Display available test folders
print("Test folders found:")
for test_name in test_data.keys():
    print(f"- {test_name}")

## 1. Trajectory Visualization

Let's visualize the trajectories from the generate_trajectory.json files. We'll plot:
- The x-y path of each trajectory
- The speed profile along the path
- The curvature profile along the path

In [ ]:
def plot_trajectories(test_data):
    """Plot the trajectories from all test folders"""
    for test_name, data in test_data.items():
        if data["trajectory"] is None:
            continue
        
        spirals, desired_speed, ego_state, behaviour, trajectories = data["trajectory"]
        
        fig, axes = plt.subplots(3, 1, figsize=(14, 18))
        
        # Plot the x-y path
        ax = axes[0]
        for i, trajectory in enumerate(trajectories):
            x = [point.get("x") for point in trajectory]
            y = [point.get("y") for point in trajectory]
            ax.plot(x, y, marker='.', label=f"Trajectory {i+1}")
            
            # Plot start and end points
            ax.plot(x[0], y[0], 'go', markersize=10, label=f"Start {i+1}" if i == 0 else "")
            ax.plot(x[-1], y[-1], 'ro', markersize=10, label=f"End {i+1}" if i == 0 else "")
        
        # Plot ego state if available
        if ego_state and "x" in ego_state and "y" in ego_state:
            ax.plot(ego_state["x"], ego_state["y"], 'bs', markersize=12, label="Ego State")
        
        ax.set_title(f"{test_name}: Trajectory Path")
        ax.set_xlabel("X position")
        ax.set_ylabel("Y position")
        ax.legend()
        ax.grid(True)
        ax.axis('equal')
        
        # Plot the velocity profile
        ax = axes[1]
        for i, trajectory in enumerate(trajectories):
            s = [point.get("s", i) for i, point in enumerate(trajectory)]
            v = [point.get("v", 0) for point in trajectory]
            ax.plot(s, v, marker='.', label=f"Trajectory {i+1}")
        
        ax.set_title(f"{test_name}: Velocity Profile")
        ax.set_xlabel("Path distance (s)")
        ax.set_ylabel("Velocity (v)")
        ax.axhline(y=desired_speed, color='r', linestyle='-', label=f"Desired speed: {desired_speed}")
        ax.legend()
        ax.grid(True)
        
        # Plot the curvature profile
        ax = axes[2]
        for i, trajectory in enumerate(trajectories):
            s = [point.get("s", i) for i, point in enumerate(trajectory)]
            kappa = [point.get("kappa", 0) for point in trajectory]
            ax.plot(s, kappa, marker='.', label=f"Trajectory {i+1}")
        
        ax.set_title(f"{test_name}: Curvature Profile")
        ax.set_xlabel("Path distance (s)")
        ax.set_ylabel("Curvature (kappa)")
        ax.legend()
        ax.grid(True)
        
        plt.tight_layout()
        plt.show()

# Plot trajectories
plot_trajectories(test_data)

## 2. Velocity Profile Analysis

Now, let's analyze the velocity profiles in more detail:
- Maximum and minimum velocities
- Acceleration and deceleration patterns
- Velocity vs. time plots

In [ ]:
def analyze_velocity_profiles(test_data):
    """Analyze velocity profiles from all trajectories"""
    for test_name, data in test_data.items():
        if data["trajectory"] is None:
            continue
        
        spirals, desired_speed, ego_state, behaviour, trajectories = data["trajectory"]
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # Plot velocity vs. time for each trajectory
        ax = axes[0, 0]
        for i, trajectory in enumerate(trajectories):
            relative_time = [point.get("relative_time", i) for i, point in enumerate(trajectory)]
            v = [point.get("v", 0) for point in trajectory]
            ax.plot(relative_time, v, marker='.', label=f"Trajectory {i+1}")
        
        ax.set_title(f"{test_name}: Velocity vs. Time")
        ax.set_xlabel("Relative Time (s)")
        ax.set_ylabel("Velocity (v)")
        ax.axhline(y=desired_speed, color='r', linestyle='-', label=f"Desired speed: {desired_speed}")
        ax.legend()
        ax.grid(True)
        
        # Plot acceleration vs. time for each trajectory
        ax = axes[0, 1]
        for i, trajectory in enumerate(trajectories):
            relative_time = [point.get("relative_time", i) for i, point in enumerate(trajectory)]
            a = [point.get("a", 0) for point in trajectory]
            ax.plot(relative_time, a, marker='.', label=f"Trajectory {i+1}")
        
        ax.set_title(f"{test_name}: Acceleration vs. Time")
        ax.set_xlabel("Relative Time (s)")
        ax.set_ylabel("Acceleration (a)")
        ax.legend()
        ax.grid(True)
        
        # Plot velocity vs. curvature for each trajectory
        ax = axes[1, 0]
        for i, trajectory in enumerate(trajectories):
            kappa = [abs(point.get("kappa", 0)) for point in trajectory]
            v = [point.get("v", 0) for point in trajectory]
            ax.scatter(kappa, v, label=f"Trajectory {i+1}")
        
        ax.set_title(f"{test_name}: Velocity vs. Curvature")
        ax.set_xlabel("Absolute Curvature |kappa|")
        ax.set_ylabel("Velocity (v)")
        ax.legend()
        ax.grid(True)
        
        # Plot velocity histogram
        ax = axes[1, 1]
        for i, trajectory in enumerate(trajectories):
            v = [point.get("v", 0) for point in trajectory]
            ax.hist(v, bins=20, alpha=0.5, label=f"Trajectory {i+1}")
        
        ax.set_title(f"{test_name}: Velocity Distribution")
        ax.set_xlabel("Velocity (v)")
        ax.set_ylabel("Frequency")
        ax.axvline(x=desired_speed, color='r', linestyle='-', label=f"Desired speed: {desired_speed}")
        ax.legend()
        ax.grid(True)
        
        plt.tight_layout()
        plt.show()

# Analyze velocity profiles
analyze_velocity_profiles(test_data)

## 3. Path Planning with Offset Goals

Let's visualize the offset goals generated by the motion planner.

In [ ]:
def plot_offset_goals(test_data):
    """Plot offset goals for each test folder"""
    for test_name, data in test_data.items():
        if data["offset_goals"] is None:
            continue
        
        goal_input, results = data["offset_goals"]
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Plot original goal
        if "x" in goal_input and "y" in goal_input:
            ax.plot(goal_input["x"], goal_input["y"], 'ro', markersize=12, label="Original Goal")
            
            # If heading is available, show it with an arrow
            if "theta" in goal_input:
                theta = goal_input["theta"]
                arrow_length = 2.0
                dx = arrow_length * np.cos(theta)
                dy = arrow_length * np.sin(theta)
                ax.arrow(goal_input["x"], goal_input["y"], dx, dy, 
                         head_width=0.5, head_length=0.7, fc='r', ec='r')
        
        # Plot offset goals
        for i, goal in enumerate(results):
            if "x" in goal and "y" in goal:
                ax.plot(goal["x"], goal["y"], 'bo', markersize=8, alpha=0.7, 
                        label=f"Offset Goal {i+1}" if i == 0 else "")
                
                # If heading is available, show it with an arrow
                if "theta" in goal:
                    theta = goal["theta"]
                    arrow_length = 1.5
                    dx = arrow_length * np.cos(theta)
                    dy = arrow_length * np.sin(theta)
                    ax.arrow(goal["x"], goal["y"], dx, dy, 
                             head_width=0.3, head_length=0.5, fc='b', ec='b', alpha=0.7)
        
        ax.set_title(f"{test_name}: Goal and Offset Goals")
        ax.set_xlabel("X position")
        ax.set_ylabel("Y position")
        ax.legend()
        ax.grid(True)
        ax.axis('equal')
        
        plt.tight_layout()
        plt.show()

# Plot offset goals
plot_offset_goals(test_data)

## 4. Cost Function Analysis

Let's visualize and analyze the cost functions evaluation for different paths.

In [ ]:
def plot_cost_functions(test_data):
    """Plot cost function analysis for each test folder"""
    for test_name, data in test_data.items():
        if not data["cost_functions"]:
            continue
        
        # Collect all cost data from this test folder
        all_spirals = []
        all_obstacles = []
        collision_costs = []
        distance_costs = []
        file_names = []
        
        for cost_func_data in data["cost_functions"]:
            file_name = cost_func_data["file"]
            file_names.append(file_name)
            
            spiral, obstacles, goal, collision_cost, distance_cost = cost_func_data["data"]
            all_spirals.append(spiral)
            all_obstacles.extend(obstacles)
            collision_costs.append(collision_cost)
            distance_costs.append(distance_cost)
        
        # Create a plot
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        
        # Plot spirals and obstacles
        ax = axes[0]
        
        # Plot unique obstacles
        obstacle_positions = set()
        for obstacle in all_obstacles:
            if "x" in obstacle and "y" in obstacle:
                pos = (obstacle["x"], obstacle["y"])
                if pos not in obstacle_positions:
                    obstacle_positions.add(pos)
                    ax.plot(obstacle["x"], obstacle["y"], 'rs', markersize=10, alpha=0.7)
        
        # Plot goal if available
        if goal and "x" in goal and "y" in goal:
            ax.plot(goal["x"], goal["y"], 'go', markersize=12, label="Goal")
        
        # Plot each spiral
        for i, spiral in enumerate(all_spirals):
            x = [point.get("x", 0) for point in spiral]
            y = [point.get("y", 0) for point in spiral]
            ax.plot(x, y, marker='.', linewidth=2, alpha=0.7, label=f"Spiral {i+1}")
        
        ax.set_title(f"{test_name}: Spirals and Obstacles")
        ax.set_xlabel("X position")
        ax.set_ylabel("Y position")
        ax.legend()
        ax.grid(True)
        ax.axis('equal')
        
        # Plot cost analysis
        ax = axes[1]
        
        # Prepare data for bar chart
        x = np.arange(len(file_names))
        width = 0.35
        
        # Plot side by side bars for collision and distance costs
        ax.bar(x - width/2, collision_costs, width, label='Collision Cost')
        ax.bar(x + width/2, distance_costs, width, label='Distance Cost')
        
        # Calculate total cost
        total_costs = [c + d for c, d in zip(collision_costs, distance_costs)]
        
        # Find minimum cost spiral
        min_cost_index = total_costs.index(min(total_costs))
        ax.plot(min_cost_index, total_costs[min_cost_index], 'g*', markersize=15, label='Minimum Cost')
        
        ax.set_xlabel('Spiral')
        ax.set_ylabel('Cost')
        ax.set_title(f"{test_name}: Cost Analysis")
        ax.set_xticks(x)
        ax.set_xticklabels([f"Spiral {i+1}" for i in range(len(file_names))])
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Detailed cost comparison table
        cost_data = {
            'Spiral': [f"Spiral {i+1}" for i in range(len(file_names))],
            'Collision Cost': collision_costs,
            'Distance Cost': distance_costs,
            'Total Cost': total_costs
        }
        cost_df = pd.DataFrame(cost_data)
        display(cost_df.style.highlight_min(axis=0, subset=['Total Cost']))

# Plot cost function analysis
plot_cost_functions(test_data)

## 5. Behavioral State Transitions

Let's analyze the behavioral planner state transitions.

In [ ]:
def plot_state_transitions(test_data):
    """Analyze behavioral planner state transitions"""
    # Collect state transition data for all tests
    state_data = []
    
    for test_name, data in test_data.items():
        if data["state_transition"] is None:
            continue
        
        ego_state, goal_input, is_junction, tl_state, sim_time, result = data["state_transition"]
        
        state_data.append({
            'Test': test_name,
            'Is Junction': is_junction,
            'TL State': tl_state,
            'Speed': ego_state.get('velocity', {}).get('magnitude', 0) if isinstance(ego_state, dict) else None,
            'Sim Time': sim_time
        })
    
    # Create a DataFrame
    if state_data:
        df = pd.DataFrame(state_data)
        
        # Create plots
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Plot junction vs. traffic light state
        ax = axes[0]
        sns.scatterplot(data=df, x='Is Junction', y='TL State', hue='Test', s=100, ax=ax)
        ax.set_title('Junction vs. Traffic Light State')
        ax.grid(True)
        
        # Plot speed vs. sim time
        ax = axes[1]
        sns.scatterplot(data=df, x='Sim Time', y='Speed', hue='Test', s=100, ax=ax)
        ax.set_title('Speed vs. Simulation Time')
        ax.grid(True)
        
        plt.tight_layout()
        plt.show()
        
        # Display the data table
        display(df)

# Plot state transitions
plot_state_transitions(test_data)

## 6. 3D Trajectory Visualization

Let's create a 3D visualization of trajectories that includes time as the third dimension.

In [ ]:
def plot_3d_trajectories(test_data):
    """Create 3D visualizations of trajectories with time as the third dimension"""
    for test_name, data in test_data.items():
        if data["trajectory"] is None:
            continue
        
        spirals, desired_speed, ego_state, behaviour, trajectories = data["trajectory"]
        
        # Create a 3D plot
        fig = plt.figure(figsize=(14, 10))
        ax = fig.add_subplot(111, projection='3d')
        
        for i, trajectory in enumerate(trajectories):
            x = [point.get("x", 0) for point in trajectory]
            y = [point.get("y", 0) for point in trajectory]
            t = [point.get("relative_time", j) for j, point in enumerate(trajectory)]
            
            # Color points by velocity
            v = [point.get("v", 0) for point in trajectory]
            
            # Create a color map based on velocity
            scatter = ax.scatter(x, y, t, c=v, cmap='viridis', s=30, alpha=0.7, 
                       label=f"Trajectory {i+1}")
        
        # Add a color bar
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Velocity (v)')
        
        ax.set_title(f"{test_name}: 3D Trajectory Visualization")
        ax.set_xlabel('X Position')
        ax.set_ylabel('Y Position')
        ax.set_zlabel('Time (s)')
        ax.legend()
        
        plt.tight_layout()
        plt.show()

# Plot 3D trajectories
plot_3d_trajectories(test_data)

## 7. Summary Statistics

Let's compute and display summary statistics for all the tests.

In [ ]:
def compute_summary_statistics(test_data):
    """Compute summary statistics for all tests"""
    # Lists to store statistics
    test_names = []
    num_trajectories = []
    avg_speeds = []
    max_speeds = []
    min_speeds = []
    avg_curvatures = []
    max_curvatures = []
    trajectory_lengths = []
    total_times = []
    
    for test_name, data in test_data.items():
        if data["trajectory"] is None:
            continue
        
        spirals, desired_speed, ego_state, behaviour, trajectories = data["trajectory"]
        
        test_names.append(test_name)
        num_trajectories.append(len(trajectories))
        
        # Compute statistics for trajectories
        test_avg_speeds = []
        test_max_speeds = []
        test_min_speeds = []
        test_avg_curvatures = []
        test_max_curvatures = []
        test_traj_lengths = []
        test_total_times = []
        
        for trajectory in trajectories:
            speeds = [point.get("v", 0) for point in trajectory]
            curvatures = [abs(point.get("kappa", 0)) for point in trajectory]
            
            test_avg_speeds.append(np.mean(speeds))
            test_max_speeds.append(max(speeds))
            test_min_speeds.append(min(speeds))
            test_avg_curvatures.append(np.mean(curvatures))
            test_max_curvatures.append(max(curvatures))
            
            # Trajectory length and time
            if trajectory:
                test_traj_lengths.append(trajectory[-1].get("s", 0))
                test_total_times.append(trajectory[-1].get("relative_time", 0))
        
        # Compute averages across all trajectories in this test
        avg_speeds.append(np.mean(test_avg_speeds) if test_avg_speeds else 0)
        max_speeds.append(max(test_max_speeds) if test_max_speeds else 0)
        min_speeds.append(min(test_min_speeds) if test_min_speeds else 0)
        avg_curvatures.append(np.mean(test_avg_curvatures) if test_avg_curvatures else 0)
        max_curvatures.append(max(test_max_curvatures) if test_max_curvatures else 0)
        trajectory_lengths.append(np.mean(test_traj_lengths) if test_traj_lengths else 0)
        total_times.append(np.mean(test_total_times) if test_total_times else 0)
    
    # Create a DataFrame with the statistics
    stats_data = {
        'Test': test_names,
        'Num Trajectories': num_trajectories,
        'Avg Speed': [f"{s:.2f}" for s in avg_speeds],
        'Max Speed': [f"{s:.2f}" for s in max_speeds],
        'Min Speed': [f"{s:.2f}" for s in min_speeds],
        'Avg Curvature': [f"{c:.4f}" for c in avg_curvatures],
        'Max Curvature': [f"{c:.4f}" for c in max_curvatures],
        'Avg Path Length': [f"{l:.2f}" for l in trajectory_lengths],
        'Avg Total Time': [f"{t:.2f}" for t in total_times]
    }
    
    stats_df = pd.DataFrame(stats_data)
    display(stats_df)
    
    # Create bar charts for key statistics
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot average speeds
    ax = axes[0, 0]
    sns.barplot(x='Test', y='Avg Speed', data=stats_df, ax=ax)
    ax.set_title('Average Speed by Test')
    ax.set_ylabel('Speed')
    ax.tick_params(axis='x', rotation=45)
    
    # Plot max curvatures
    ax = axes[0, 1]
    sns.barplot(x='Test', y='Max Curvature', data=stats_df, ax=ax)
    ax.set_title('Maximum Curvature by Test')
    ax.set_ylabel('Curvature')
    ax.tick_params(axis='x', rotation=45)
    
    # Plot trajectory lengths
    ax = axes[1, 0]
    sns.barplot(x='Test', y='Avg Path Length', data=stats_df, ax=ax)
    ax.set_title('Average Path Length by Test')
    ax.set_ylabel('Path Length')
    ax.tick_params(axis='x', rotation=45)
    
    # Plot total times
    ax = axes[1, 1]
    sns.barplot(x='Test', y='Avg Total Time', data=stats_df, ax=ax)
    ax.set_title('Average Total Time by Test')
    ax.set_ylabel('Time (s)')
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

# Compute and display summary statistics
compute_summary_statistics(test_data)

## Conclusion

This analysis provides insights into the autonomous vehicle planning system behavior across different test scenarios. We've visualized:

1. Trajectories and path planning in 2D and 3D
2. Velocity profiles and their relationship with curvature
3. Offset goal generation for motion planning
4. Cost function evaluations for different path options
5. Behavioral state transitions

These visualizations help understand how the planning system generates paths, selects velocities, evaluates costs, and transitions between behavioral states.